In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

In [2]:
import pandas as pd
from datasets import Dataset, ClassLabel
import numpy as np
import random
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader
import gc
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import pipeline
from datasets import concatenate_datasets

In [3]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-MiniLM2-L6-H768",
    "typeform/distilbert-base-uncased-mnli",
    "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary",
    "tasksource/deberta-small-long-nli",
    "MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33",
    "cmarkea/distilcamembert-base-nli",
    "MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33",
    "MoritzLaurer/deberta-v3-base-zeroshot-v1"
]

LABELS = ["met palliative care needs", "unmet palliative care needs"]
LABEL_TO_NUM = {"met palliative care needs": 0, "unmet palliative care needs": 1}
LABEL_EXPANSIONS = {
    "met palliative care needs": [
        "palliative care needs are adequately met",
        "symptoms and care needs are well managed",
        "the patient is receiving appropriate palliative care",
        "care needs are sufficiently addressed"
    ],
    "unmet palliative care needs": [
        "palliative care needs are not adequately met",
        "symptoms or care needs are poorly managed",
        "the patient requires additional palliative care support",
        "care needs are not sufficiently addressed"
    ]
}

HYPOTHESIS_TEMPLATES = [
    # Generic hypothesis templates.
    "This example is about {}.",
    # Generic medical hypothesis templates. 
    "The patient's palliative care needs are {}.",
    "Overall, the patient's care needs are {}.",
    "From this note, it can be inferred that {}.",
    "This clinical note indicates that the patient's care needs are {}.",
    "Based on this note, the patient's care needs are {}.",
    # Clinical language.
    "The patient's condition suggests that {}.",
    "This note suggests that the patient is experiencing {}.",
    "The patient's current situation reflects {}.",
    # Care quality framing.
    "The patient's symptoms and care needs are {}.",
    "The patient's care is {}.",
    # Documentation style focus.
    "This note indicates a situation where {}.",
    "The note documents that {}.",
    "The record indicates that {}.",
    "The clinical documentation suggests that {}.",
]

In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [5]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

# Map dataset labels to 1s or 0s.
label_map = {
    "met palliative care needs": 0,
    "unmet palliative care needs": 1
}

dataset = dataset.map(lambda x: {"label": label_map[f"{x["needs"]} palliative care needs"]})


label_feature = ClassLabel(names=["met palliative care needs", "unmet palliative care needs"])
dataset = dataset.cast_column("label", label_feature)

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [6]:
# Split the dataset. (80/10/10)
dataset = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE, stratify_by_column="label")

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_split = temp_dataset.train_test_split(test_size=0.5, seed=RANDOM_STATE, stratify_by_column="label")

val_dataset = temp_split["train"]
test_dataset = temp_split["test"]

all_dataset = concatenate_datasets([train_dataset, val_dataset, test_dataset])

# Check distributions.
def check_distribution(dataset, name):
    df = dataset.to_pandas()
    counts = df["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_dataset, "Train")
check_distribution(val_dataset, "Validation")
check_distribution(test_dataset, "Test")
check_distribution(all_dataset, "All Data")

Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Validation distribution:
label
0    0.50173
1    0.49827
Name: proportion, dtype: float64
Test distribution:
label
0    0.502591
1    0.497409
Name: proportion, dtype: float64
All Data distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64


In [7]:
# Make evaluation function.
def evaluate_model(preds, labels):
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [8]:
results = {}
device = 0 if torch.cuda.is_available() else -1

def run_models(dataset, hypo_temp="This clinical note indicates that {}."):
    for model_name in ZERO_SHOT_MODELS:
        print(f"\nRunning zero-shot model: {model_name}")

        classifier = pipeline(
            "zero-shot-classification",
            model=model_name,
            device=device,
        )

        texts = [ex["report"] for ex in dataset]
        true_labels = [ex["label"] for ex in dataset]

        # Flatten expanded labels.
        flat_labels = []
        label_map = {}

        for base_label, expansions in LABEL_EXPANSIONS.items():
            for exp in expansions:
                flat_labels.append(exp)
                label_map[exp] = base_label

        # Run model.
        outputs = classifier(
            texts,
            candidate_labels=flat_labels,
            hypothesis_template=hypo_temp,
            batch_size=8
        )

        preds = []

        # Aggregate scores per base label.
        for o in outputs:
            scores_by_class = {k: 0.0 for k in LABEL_EXPANSIONS.keys()}

            for label, score in zip(o["labels"], o["scores"]):
                base_label = label_map[label]
                scores_by_class[base_label] += score

            # Pick best aggregated label.
            pred = max(scores_by_class, key=scores_by_class.get)
            preds.append(pred)
        numeric_preds = [0 if pred == "met palliative care needs" else 1 for pred in preds]
        metrics = evaluate_model(numeric_preds, true_labels)

        # Inner dictionary creation.
        if model_name not in results:
            results[model_name] = {}  

        results[model_name][hypo_temp] = metrics

        # Output.
        print("----------------------------------------------")
        print("--------------- Metric Results ---------------")
        print(f"\nModel: {model_name} \nHypothesis Template: {hypo_temp}")
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")

        print("--------------- Confusion Matrix ---------------")
        print(confusion_matrix(true_labels, numeric_preds))

        print("--------------- Classification Report ---------------")
        print(classification_report(true_labels, numeric_preds))

        print("----------------------------------------------\n\n\n")

        # Free memory.
        del classifier
        torch.cuda.empty_cache()
        gc.collect()

    return results


In [9]:
# Run all hypothesis templates.
all_results = []
for template in HYPOTHESIS_TEMPLATES:
    all_results.append(run_models(all_dataset, hypo_temp=template))

# Save in DataFrame.
rows = []

for model_name, templates in results.items():
    for hypo_template, metrics in templates.items():
        row = {"model": model_name, "hypothesis_template": hypo_template}
        row.update(metrics)
        rows.append(row)

# Save DataFrame.
all_df = pd.DataFrame(rows)
all_df.to_csv('./all_data_all_zero_shot_results.csv', index=False)


Running zero-shot model: cross-encoder/nli-MiniLM2-L6-H768


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


----------------------------------------------
--------------- Metric Results ---------------

Model: cross-encoder/nli-MiniLM2-L6-H768 
Hypothesis Template: This example is about {}.
accuracy: 0.6599
precision: 0.7747
recall: 0.4467
f1: 0.5666
--------------- Confusion Matrix ---------------
[[2530  374]
 [1593 1286]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.61      0.87      0.72      2904
           1       0.77      0.45      0.57      2879

    accuracy                           0.66      5783
   macro avg       0.69      0.66      0.64      5783
weighted avg       0.69      0.66      0.64      5783

----------------------------------------------




Running zero-shot model: typeform/distilbert-base-uncased-mnli


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

----------------------------------------------
--------------- Metric Results ---------------

Model: cmarkea/distilcamembert-base-nli 
Hypothesis Template: The clinical documentation suggests that {}.
accuracy: 0.5649
precision: 0.6637
recall: 0.2556
f1: 0.3691
--------------- Confusion Matrix ---------------
[[2531  373]
 [2143  736]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.54      0.87      0.67      2904
           1       0.66      0.26      0.37      2879

    accuracy                           0.56      5783
   macro avg       0.60      0.56      0.52      5783
weighted avg       0.60      0.56      0.52      5783

----------------------------------------------




Running zero-shot model: MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

----------------------------------------------
--------------- Metric Results ---------------

Model: MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33 
Hypothesis Template: The clinical documentation suggests that {}.
accuracy: 0.7619
precision: 0.7860
recall: 0.7169
f1: 0.7499
--------------- Confusion Matrix ---------------
[[2342  562]
 [ 815 2064]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.74      0.81      0.77      2904
           1       0.79      0.72      0.75      2879

    accuracy                           0.76      5783
   macro avg       0.76      0.76      0.76      5783
weighted avg       0.76      0.76      0.76      5783

----------------------------------------------




Running zero-shot model: MoritzLaurer/deberta-v3-base-zeroshot-v1


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

----------------------------------------------
--------------- Metric Results ---------------

Model: MoritzLaurer/deberta-v3-base-zeroshot-v1 
Hypothesis Template: The clinical documentation suggests that {}.
accuracy: 0.8395
precision: 0.8685
recall: 0.7985
f1: 0.8321
--------------- Confusion Matrix ---------------
[[2556  348]
 [ 580 2299]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.82      0.88      0.85      2904
           1       0.87      0.80      0.83      2879

    accuracy                           0.84      5783
   macro avg       0.84      0.84      0.84      5783
weighted avg       0.84      0.84      0.84      5783

----------------------------------------------





In [10]:
all_df.sort_values(by='accuracy', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
108,MoritzLaurer/deberta-v3-base-zeroshot-v1,"From this note, it can be inferred that {}.",0.844717,0.857710,0.824939,0.841006
109,MoritzLaurer/deberta-v3-base-zeroshot-v1,This clinical note indicates that the patient'...,0.842469,0.850178,0.829802,0.839866
118,MoritzLaurer/deberta-v3-base-zeroshot-v1,The record indicates that {}.,0.841086,0.855072,0.819729,0.837028
119,MoritzLaurer/deberta-v3-base-zeroshot-v1,The clinical documentation suggests that {}.,0.839530,0.868530,0.798541,0.832067
111,MoritzLaurer/deberta-v3-base-zeroshot-v1,The patient's condition suggests that {}.,0.837628,0.874228,0.787079,0.828368
...,...,...,...,...,...,...
78,cmarkea/distilcamembert-base-nli,"From this note, it can be inferred that {}.",0.546948,0.704581,0.154915,0.253986
88,cmarkea/distilcamembert-base-nli,The record indicates that {}.,0.545046,0.628099,0.211184,0.316090
80,cmarkea/distilcamembert-base-nli,"Based on this note, the patient's care needs a...",0.542798,0.679389,0.154568,0.251839
86,cmarkea/distilcamembert-base-nli,This note indicates a situation where {}.,0.537610,0.646220,0.157346,0.253073


In [11]:
all_df.sort_values(by='precision', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
45,tasksource/deberta-small-long-nli,This example is about {}.,0.702231,0.903136,0.450156,0.600834
105,MoritzLaurer/deberta-v3-base-zeroshot-v1,This example is about {}.,0.703095,0.895238,0.457103,0.605197
57,tasksource/deberta-small-long-nli,The note documents that {}.,0.778662,0.888673,0.634943,0.740681
52,tasksource/deberta-small-long-nli,This note suggests that the patient is experie...,0.767422,0.883500,0.613755,0.724329
47,tasksource/deberta-small-long-nli,"Overall, the patient's care needs are {}.",0.782985,0.882298,0.650920,0.749151
...,...,...,...,...,...,...
32,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,"Overall, the patient's care needs are {}.",0.636867,0.581605,0.964224,0.725562
35,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,"Based on this note, the patient's care needs a...",0.598132,0.556231,0.953456,0.702585
39,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The patient's symptoms and care needs are {}.,0.570465,0.538507,0.959361,0.689810
61,MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-a...,The patient's palliative care needs are {}.,0.553865,0.527456,0.997569,0.690053


In [12]:
all_df.sort_values(by='recall', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
31,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The patient's palliative care needs are {}.,0.522739,0.510567,0.998611,0.675676
61,MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-a...,The patient's palliative care needs are {}.,0.553865,0.527456,0.997569,0.690053
41,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,This note indicates a situation where {}.,0.651392,0.588987,0.992011,0.739130
43,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The record indicates that {}.,0.707764,0.633087,0.982286,0.769943
44,MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli...,The clinical documentation suggests that {}.,0.692720,0.621903,0.976381,0.759832
...,...,...,...,...,...,...
84,cmarkea/distilcamembert-base-nli,The patient's symptoms and care needs are {}.,0.555248,0.712310,0.178882,0.285952
75,cmarkea/distilcamembert-base-nli,This example is about {}.,0.547640,0.680384,0.172282,0.274945
86,cmarkea/distilcamembert-base-nli,This note indicates a situation where {}.,0.537610,0.646220,0.157346,0.253073
78,cmarkea/distilcamembert-base-nli,"From this note, it can be inferred that {}.",0.546948,0.704581,0.154915,0.253986


In [13]:
all_df.sort_values(by='f1', ascending=False)

,model,hypothesis_template,accuracy,precision,recall,f1
108,MoritzLaurer/deberta-v3-base-zeroshot-v1,"From this note, it can be inferred that {}.",0.844717,0.857710,0.824939,0.841006
109,MoritzLaurer/deberta-v3-base-zeroshot-v1,This clinical note indicates that the patient'...,0.842469,0.850178,0.829802,0.839866
118,MoritzLaurer/deberta-v3-base-zeroshot-v1,The record indicates that {}.,0.841086,0.855072,0.819729,0.837028
64,MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-a...,This clinical note indicates that the patient'...,0.818952,0.770686,0.905870,0.832828
119,MoritzLaurer/deberta-v3-base-zeroshot-v1,The clinical documentation suggests that {}.,0.839530,0.868530,0.798541,0.832067
...,...,...,...,...,...,...
84,cmarkea/distilcamembert-base-nli,The patient's symptoms and care needs are {}.,0.555248,0.712310,0.178882,0.285952
75,cmarkea/distilcamembert-base-nli,This example is about {}.,0.547640,0.680384,0.172282,0.274945
78,cmarkea/distilcamembert-base-nli,"From this note, it can be inferred that {}.",0.546948,0.704581,0.154915,0.253986
86,cmarkea/distilcamembert-base-nli,This note indicates a situation where {}.,0.537610,0.646220,0.157346,0.253073
